In [ ]:
import ctypes
import numpy as np
import time
import cv2
import gc
import os
import zlib
from sdlarch_rl import make
import pygame
import gzip

env = make("SuperStreetFighterIV-3DS")
# env = make("GranTurismo3-Ps2")
#env = make("NewSuperMarioBros-Wii", env_id=1)
# env = make("VirtuaTennis-DC")
# env = make("CrazyTaxi-DC")
# env = make("SuperMario64-N64")
# env = make("SuperSmashBrosBrawl-Wii")
# env = make("GodOfWar-PSP")
# env = make("YoshiIsland-NDS")

obs, info = env.reset()

count = 0
global initial_state
initial_state = None

pygame.init()

SCREEN_WIDTH = 640
SCREEN_HEIGHT = 480

if os.name == 'nt':
    SCREEN_WIDTH = 768*3
    SCREEN_HEIGHT = 640*3
    
window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()

paused = False

# Wii games
INVERT_AXIS=False

framerate = env.unwrapped.em.get_frame_rate()

print("framerate: ", framerate)

frame_time = 1.0 / framerate
last_time = time.time()

while True:
    pygame.event.pump()
        
    keys = pygame.key.get_pressed()

    # Pause
    if keys[pygame.K_BACKSPACE]:
        paused = not paused
        print("== PAUSED ==" if paused else "== RETURNING ==")
        time.sleep(0.3)  # debounce

    if keys[pygame.K_s]:
        paused = not paused

        if paused:
            state = env.unwrapped.em.get_state()
            with gzip.open("default.state", "wb") as f:
                f.write(state)

    if paused:
        clock.tick(10)
        continue
    
    action = np.zeros(16, dtype=np.uint8)

    if keys[pygame.K_UP]:
        if INVERT_AXIS:
            action[7] = 1
        else:
            action[4] = 1
    if keys[pygame.K_DOWN]:
        if INVERT_AXIS:
            action[6] = 1
        else:
            action[5] = 1
    if keys[pygame.K_LEFT]:
        if INVERT_AXIS:
            action[4] = 1
        else:
            action[6] = 1
    if keys[pygame.K_RIGHT]:
        if INVERT_AXIS:
            action[5] = 1
        else:
            action[7] = 1
    if keys[pygame.K_c]:
        action[1] = 1
    if keys[pygame.K_x]:
        action[0] = 1
    if keys[pygame.K_RETURN]:
        action[3] = 1
    if keys[pygame.K_l]:
        action[11] = 1

    # clock.tick(60)
    
    img, rew, done, _, info = env.step(action)

    font = cv2.FONT_HERSHEY_SIMPLEX
            
    # org
    org = (400, 50)
    
    # fontScale
    fontScale = 1
     
    # Blue color in BGR
    color = (255, 0, 0)
    
    # Line thickness of 2 px
    thickness = 2

    img = cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT))
     
    # Using cv2.putText() method
    img = cv2.putText(img, "Paused: " + str(paused + 1), org, font, 
                       fontScale, color, thickness, cv2.LINE_AA)


    surface = pygame.surfarray.make_surface(np.transpose(img, (1, 0, 2)))


    window.blit(surface, (0, 0))
    pygame.display.update()

    count += 1

    # break

    if count % 1000 == 0:
        # env.reset()
        pass

    

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


statename is None setting to default state
State file not found: D:\projects\sdlarch-rl\sdlarch_rl\roms\SuperStreetFighterIV-3DS\default.state. Starting without initial state.
framerate:  60.0
